In [ ]:
##this is a small function to identify the change in boolean state in an array
def a_guest(x):
    b = np.zeros_like(x, dtype=bool)
    i = np.argmax(x, axis=1)
    b[np.arange(i.size), i] = np.logical_or.reduce(x, axis=1)
    return b

In [ ]:
def deceleration_detection(v_array = [], *args):
    V0=v_array[:,init_time*frame_rate:int((init_time+0.5)*frame_rate),0]
    meanV0=np.mean(V0, axis=1)
    stdV0=np.std(V0, axis=1, ddof=1)
    deceleration_threshold=meanV0-2*stdV0
    velocity_of_interest=v_array[:,int((init_time+0.5)*frame_rate):int((init_time+presence_duration+0.5)*frame_rate),0]
    dece_big=np.reshape(np.repeat(deceleration_threshold,presence_duration*frame_rate,axis=0),(len(deceleration_threshold),presence_duration*frame_rate))
    dec=np.less(dece_big,velocity_of_interest)
    rev=np.fliplr(dec)
    dev_onset=np.where(a_guest(rev)==True)
    velocity_of_interest.shape[1]-dev_onset[1]
    return dev_onset

In [ ]:
thisStim="c2f_f_sloom"
if analyse_every_trial==True:
    stim_to_analyse=stim_df.loc[:,'stim_onset']
    stim_name_list=np.array(stim_df["stim_type"])
#elif: 'myVar' in locals():
#stim_type=["m2b_f_sloom","m2b_s_sloom","m2f_f_sloom","m2f_s_sloom","m2m_f_loom","m2m_s_loom"]
else:
    #stim_to_analyse=stim_df.loc[np.where(stim_df.loc[:,'stim_type']==stim_of_interest),"stim_onset"]
    #stim_to_analyse=stim_df.loc[np.where((stim_df.loc[:,'stim_type']== "f2b_l_sweep") | (stim_df.loc[:,'stim_type']== "f2b_r_sweep")),"stim_onset"]
    #stim_to_analyse=stim_df.loc[np.where((stim_df.loc[:,'stim_type']== "b2f_l_sweep") | (stim_df.loc[:,'stim_type']== "b2f_r_sweep")),"stim_onset"]
    #stim_to_analyse=stim_df.loc[np.where((stim_df.loc[:,'stim_type']== "b2f_l_sloom") | (stim_df.loc[:,'stim_type']== "b2f_r_sloom")),"stim_onset"]
    #stim_to_analyse=stim_df.loc[np.where((stim_df.loc[:,'stim_type']== "c2b_f_sloom") | (stim_df.loc[:,'stim_type']== "m2b_s_sloom")),"stim_onset"]
    stim_to_analyse=stim_df.loc[np.where((stim_df.loc[:,'stim_type']== thisStim)),"stim_onset"]
    #stim_to_analyse=stim_df.loc[np.where((stim_df.loc[:,'stim_type']== "c2c_s_loom")),"stim_onset"]
    stim_name_list=stim_df.loc[stim_to_analyse.index,'stim_type'].values
if len(stim_to_analyse) ==0:
    print('no_required_stim_type')
    #continue

if re.search('_f_', thisStim):##create this condition so that stim_duration became dynamics
    stim_duration=3
    presence_duration=1
    sweep_duration=1
    half_presentation=0.5
elif re.search('_s_', thisStim):
    stim_duration=4
    presence_duration=1
    half_presentation=0.5
    sweep_duration=2
else:
    print('something wrong with the stim list')

# cum_travel_distance=np.cumsum(np.sqrt(np.add(np.square(x_dif), np.square(y_dif))),axis=1)
# init_time=abs(analysis_window[0])
# end_time=abs(analysis_window[0])+stim_duration
# travel_distance_prior_stim=cum_travel_distance[:,init_time*frame_rate,0]
# travel_distance_post_stim=cum_travel_distance[:,end_time*frame_rate,0]
# travel_distance_during_stim=travel_distance_post_stim-travel_distance_prior_stim
# normalise_travel_distance=np.divide(travel_distance_post_stim,travel_distance_prior_stim)       
# travel_distance_prior_stim_all.append(travel_distance_prior_stim)
# travel_distance_tbt.append(cum_travel_distance)
# mean_travel_distance_during_stim.append(np.mean(travel_distance_during_stim))
# mean_normalised_distance_during_stim.append(np.mean(normalise_travel_distance))

grey_onset=stim_to_analyse+np.ones(len(stim_to_analyse),dtype=int)*analysis_window[0]*frame_rate
analysie_window_tile=np.tile(np.arange(window_length*frame_rate),(len(grey_onset),1))
index_points=np.array(grey_onset).reshape(len(grey_onset),1)+analysie_window_tile
x=x_all[index_points]
y=y_all[index_points]
x_dif=np.diff(x,axis=1)
y_dif=np.diff(y,axis=1)


travel_distance_fbf=np.sqrt(np.add(np.square(x_dif), np.square(y_dif)))
velocity=travel_distance_fbf*frame_rate
cum_travel_distance=np.cumsum(np.sqrt(np.add(np.square(x_dif), np.square(y_dif))),axis=1)
init_time=abs(analysis_window[0])
end_time=abs(analysis_window[0])+stim_duration
travel_distance_prior_stim=cum_travel_distance[:,init_time*frame_rate,0]
travel_distance_post_stim=cum_travel_distance[:,end_time*frame_rate,0]


travel_distance_after_stim_presence=cum_travel_distance[:,(init_time+presence_duration)*frame_rate,0]
travel_distance_after_stim_sweep=cum_travel_distance[:,(init_time+presence_duration+sweep_duration)*frame_rate,0]
travel_distance_during_stim_presence=travel_distance_after_stim_presence-travel_distance_prior_stim
travel_distance_during_stim_sweep=travel_distance_after_stim_sweep-travel_distance_after_stim_presence
normalised_distance_change=np.divide(travel_distance_during_stim_sweep,travel_distance_during_stim_presence)

mean_travel_distance_post_stim=np.mean(travel_distance_post_stim)
travel_distance_during_stim=travel_distance_post_stim-travel_distance_prior_stim
normalise_travel_distance=np.divide(travel_distance_post_stim,travel_distance_prior_stim)  
mean_normalise_travel_distance=np.mean(normalise_travel_distance)


travel_distance_1sec_prior_stim=cum_travel_distance[:,init_time*frame_rate,0]-cum_travel_distance[:,(init_time-1)*frame_rate,0]
running_trial=travel_distance_1sec_prior_stim>5
running_velocity=velocity[running_trial,:]
runnning_trial_distance_change=normalised_distance_change[running_trial]
travel_distance_1sec_post_stim=cum_travel_distance[running_trial,(init_time+1)*frame_rate,0]-cum_travel_distance[running_trial,init_time*frame_rate,0]
velocity_type1=running_velocity[travel_distance_1sec_post_stim>5,:]
velocity_type2=running_velocity[travel_distance_1sec_post_stim<5,:]
runnning_trial_distance_change_type1=runnning_trial_distance_change[travel_distance_1sec_post_stim>5]
runnning_trial_distance_change_type2=runnning_trial_distance_change[travel_distance_1sec_post_stim<5]
stimulus_order.append(sweep_duration)
travel_distance_prior_stim_all.append(travel_distance_prior_stim)
travel_distance_tbt.append(cum_travel_distance)
distance_change_t1.append(runnning_trial_distance_change_type1)
distance_change_t2.append(runnning_trial_distance_change_type1)
## please tell me why I wanted to take mean here
mean_travel_distance_during_stim.append(np.mean(travel_distance_during_stim))
mean_normalised_distance_during_stim.append(np.mean(normalise_travel_distance))



#travel_distance=np.sum(np.sqrt(np.add(np.square(x_dif), np.square(y_dif))),axis=1)
vx=df.loc[:, ['delta rotation vector lab x']].values
velocity_x=vx[index_points]
vz=df.loc[:, ['delta rotation vector lab z']].values
velocity_z=vz[index_points]
#instant_v=df.loc[:, ['instant speed']].values
##this is a small test for how to quantify velocity
#velocity=instant_v[index_points]
velocity_rolling=runmean_v[index_points]
tp=np.linspace(0,window_length,velocity.shape[1])
tp_tile=np.tile(tp,(velocity.shape[0],1))

In [ ]:
detected_epochs=deceleration_detection(velocity_type1)
de_test=frame_rate*presence_duration-detected_epochs[1]
dev_timing=de_test/frame_rate+0.5*presence_duration+init_time
dev_timing_y=10*np.ones_like(dev_timing)
mean_velocity=np.mean(velocity_type1[:,:,0], axis=0)
sem_velocity=np.std(velocity_type1[:,:,0], axis=0, ddof=1) / np.sqrt(velocity_type1[:,:,0].shape[0])
v_base=np.median(velocity_type1[:,init_time*frame_rate:(init_time+1)*frame_rate,0],axis=1)
v_interest=velocity_type1[:,:,0]
normalise_v=v_interest.T/v_base
mean_velocity_norm=np.mean(normalise_v.T, axis=0)
sem_velocity_norm=np.std(normalise_v.T, axis=0, ddof=1) / np.sqrt(normalise_v.T.shape[0])
mean_tile=np.mean(tp_tile, axis=0)
v_add_sem = mean_velocity + sem_velocity
v_min_sem = mean_velocity - sem_velocity
v_add_sem_norm = mean_velocity_norm + sem_velocity_norm
v_min_sem_norm = mean_velocity_norm - sem_velocity_norm
fig, (ax,ax2)= plt.subplots(nrows=1, ncols=2, figsize=(18,7),tight_layout=True)
# ax=fig.add_axes([0.1, 0.1, 0.8, 0.8]) # main axes
ax.plot(mean_tile,mean_velocity)
ax.plot(dev_timing, dev_timing_y, 'o', color='r')
ax.fill_between(mean_tile,v_add_sem,v_min_sem, alpha=0.5, linewidth=0)
ax.axvline(x=abs(analysis_window[0]), color='black', linestyle='-', label='Stim start',linewidth=1)
ax.axvline(x=(abs(analysis_window[0])+stim_duration), color='black', linestyle='--', label='Stim end',linewidth=1)
ax.set_ylim(0, 60)
ax.set_yticks([0,30,60])
ax.set_yticklabels(['0','30','60'])
ax.set_xticks([1,2,3,4,5,6])
ax.set_xticklabels(['-1','0','1','2','3','4'])
plot_title=f'average response from {velocity_type1[:,:,0].shape[0]} trials'
ax.set_title(plot_title)

ax2.plot(mean_tile,mean_velocity_norm)
ax2.fill_between(mean_tile,v_add_sem_norm,v_min_sem_norm, alpha=0.5, linewidth=0)
ax2.axvline(x=abs(analysis_window[0]), color='black', linestyle='-', label='Stim start',linewidth=1)
ax2.axvline(x=(abs(analysis_window[0])+stim_duration), color='black', linestyle='--', label='Stim end',linewidth=1)
ax2.set_ylim(0.01, 100)
ax2.set_yscale('log')
ax2.set_xticks([1,2,3,4,5,6])
ax2.set_xticklabels(['-1','0','1','2','3','4'])
plot_title=f'average response from normalised velocities'
ax2.set_title(plot_title)
plot_file=f'stim_{stim_name_list[0]}_velocity_run_trials_from_{str(analysis_window[0])}_to_{str(analysis_window[1])}s.png'
if save_fig == True:
    plt.savefig(thisDir+plot_file)

In [ ]:
for thisVelocity in range(0, velocity_type1.shape[0]):
    fig, (ax,ax2)= plt.subplots(nrows=1, ncols=2, figsize=(18,7),tight_layout=True)
    # ax=fig.add_axes([0.1, 0.1, 0.8, 0.8]) # main axes
    ax.plot(mean_tile,velocity_type1[thisVelocity,:,0])
    if thisVelocity in detected_epochs[0]:
        thisTiming=dev_timing[detected_epochs[0]==thisVelocity]
        thisValue=30*np.ones_like(thisTiming)
        
        ax.plot(thisTiming, thisValue, 'o', color='r')
    #ax.fill_between(mean_tile,v_add_sem,v_min_sem, alpha=0.5, linewidth=0)
    ax.axvline(x=abs(analysis_window[0]), color='black', linestyle='-', label='Stim start',linewidth=1)
    ax.axvline(x=(abs(analysis_window[0])+stim_duration), color='black', linestyle='--', label='Stim end',linewidth=1)
    ax.set_ylim(0, 60)
    ax.set_yticks([0,30,60])
    ax.set_yticklabels(['0','30','60'])
    ax.set_xticks([1,2,3,4,5,6])
    ax.set_xticklabels(['-1','0','1','2','3','4'])
    plot_title=f'average response from {velocity_type1[:,:,0].shape[0]} trials'
    ax.set_title(plot_title)

    ax2.plot(mean_tile,mean_velocity_norm)
    ax2.fill_between(mean_tile,v_add_sem_norm,v_min_sem_norm, alpha=0.5, linewidth=0)
    ax2.axvline(x=abs(analysis_window[0]), color='black', linestyle='-', label='Stim start',linewidth=1)
    ax2.axvline(x=(abs(analysis_window[0])+stim_duration), color='black', linestyle='--', label='Stim end',linewidth=1)
    ax2.set_ylim(0.01, 100)
    ax2.set_yscale('log')
    ax2.set_xticks([1,2,3,4,5,6])
    ax2.set_xticklabels(['-1','0','1','2','3','4'])
    plot_title=f'average response from normalised velocities'
    ax2.set_title(plot_title)
    plot_file=f'stim_{stim_name_list[0]}_velocity_run_trials_from_{str(analysis_window[0])}_to_{str(analysis_window[1])}s.png'
    #if save_fig == True:
        #plt.savefig(thisDir+plot_file)

In [ ]:
##this is a good cell to plot average response plue tbt response
mean_velocity=np.mean(velocity, axis=0)
sem_velocity=np.std(velocity, axis=0, ddof=1) / np.sqrt(velocity.shape[0])
mean_tile=np.mean(tp_tile, axis=0)
v_add_sem = mean_velocity + sem_velocity
v_min_sem = mean_velocity - sem_velocity
fig = plt.figure()
ax=fig.add_axes([0.1, 0.1, 0.8, 0.8]) # main axes
ax.plot(mean_tile,mean_velocity)
for thisRow in range(0,index_points.shape[0]):
    ax.plot(mean_tile,velocity[thisRow,:],'k',alpha=0.5,lw=0.5)
#ax.fill_between(mean_tile,v_add_sem[:,0],v_min_sem[:,0], alpha=0.5, linewidth=0)
ax.axvline(x=abs(analysis_window[0]), color='black', linestyle='-', label='Second Line')
ax.axvline(x=(abs(analysis_window[0])+stim_duration), color='black', linestyle='--', label='Second Line')
#plt.ylim(0, 0.05)
#ax.set_yticks([0,0.025,0.05])
#ax.set_yticklabels(['0','0.025','0.05'])
ax.set_xticks([0,2,4,6])
ax.set_xticklabels(['-2','0','2','4'])
plot_file=f'stim_{stim_name_list[0]}_mean_velocity_from_{str(analysis_window[0])}_to_{str(analysis_window[1])}s.png'
#plt.savefig(thisDir+plot_file)